# Week 5: Dynamic Hedging Simulation

This notebook further develops dynamic hedging simulated from [Week 4](../week4_greeks/greeks.ipynb).

Getting simulation instances from Week 1's GBM simulation, the goal is to input the different GBM walks into a dynamic hedging simulation.

In [ ]:
from scipy.stats import norm
from math import log, sqrt, exp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Setup inputs
s_0 = 49
k = 50
r = 0.05
mu = 0.15
sigma = 0.30
T = 20/52        # 20 weeks in years
N = 20     # one step per week
dt = T / N # = 1/52
num_paths = 1000
n_options = 100000

# Generating random draws
Z = np.random.standard_normal((num_paths, N))

#  Building price paths in GBM
prices = np.zeros ((num_paths, N+1))
prices[:, 0] = s_0         
for t in range(1, N+1):
            prices[:, t] = prices[:, t-1] * np.exp((mu - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * Z[:, t-1])

# Function for d1 and d2
def d_1(sigma, s_0, k, r, T):
    d_1 = (log(s_0/k)+(r+(sigma**2)/2)*T)/((sigma)*sqrt(T))
    return d_1

def d_2(sigma, s_0, k, r, T): 
    return d_1(sigma, s_0, k, r, T) - (sigma)*sqrt(T)

# Identifying option prices using BSM
def compute_delta_hedge(sigma, S, k, r, T_remaining):
    if T_remaining <= 0:
        return 1.0 if S > k else 0.0
    return norm.cdf(d_1(sigma, S, k, r, T_remaining))

# BSM call pricer only
def bsm(sigma, s_0, k, r, T):
    d1 = d_1(sigma, s_0, k, r, T)
    d2 = d_2(sigma, s_0, k, r, T)
    opt_price = s_0 * norm.cdf(d1) - k * exp(-r * T) * norm.cdf(d2)
    return opt_price

# Compute options cost with BSM:
bsm_price = bsm(sigma, s_0, k, r, T) * n_options

# Build pnl list
pnl_list = []

for i in range(num_paths):
    stock_prices = prices[i]
    
    hedge_account = bsm_price  # start with premium collected
    shares_held = 0.0
    
    for t in range(N):
        # Grow account by one period of interest
        hedge_account *= exp(r * dt)
        
        # Compute new delta
        T_remaining = T - t * dt
        new_delta = compute_delta_hedge(sigma, stock_prices[t], k, r, T_remaining)
        
        # Rebalance: buy/sell shares
        shares_to_trade = (new_delta - shares_held) * n_options
        hedge_account -= shares_to_trade * stock_prices[t]
        shares_held = new_delta * n_options
    
    # Expiry: liquidate shares, pay option payoff
    hedge_account += shares_held * stock_prices[-1]
    option_payoff = max(stock_prices[-1] - k, 0) * n_options
    hedge_account -= option_payoff
    
    pnl_list.append(hedge_account)

i = 0
stock_prices = prices[i]
hedge_account = bsm_price
shares_held = 0.0

for t in range(3):
    hedge_account *= exp(r * dt)
    T_remaining = T - t * dt
    new_delta = compute_delta_hedge(sigma, stock_prices[t], k, r, T_remaining)
    shares_to_trade = (new_delta - shares_held) * n_options
    hedge_account -= shares_to_trade * stock_prices[t]
    shares_held = new_delta * n_options
    print(f"t={t}: delta={new_delta:.4f}, shares_to_trade={shares_to_trade:,.2f}, hedge_account={hedge_account:,.2f}")






BSM price: 360,973.07
Cum cost: 2,330,930.44
Option payoff: 0.00
P&L: -1,977,256.44
cost first 3: [np.float64(2621416.6613105764), np.float64(45173.79048540736), np.float64(-315588.96468982025)]
cum_cost first 3: [np.float64(2621416.6613105764), np.float64(2669155.7141781347), np.float64(2355830.8827909287)]
len cost: 20
delta[0]: 0.5350
shares[0]: 53,498.30
stock_prices[0]: 49.00
